# 15 — Grand Ensemble: RidgeCV Meta-Learner on All OOF Predictions

Pools out-of-fold predictions from every base model into a single RidgeCV meta-learner.

| Model | OOF RAE | OOF-residual ρ vs LGBM_aug |
|---|---|---|
| LGBM_base | 0.5600 | 0.937 |
| LGBM_aug | 0.5633 | — |
| kNN (k=20) | 0.7341 | 0.686 |
| ChemBERTa-MLM (nb13) | 0.6782 | **0.728** (most diverse!) |
| ChemBERTa-MTR (nb14) | 0.5993 | 0.835 |
| **Meta-11** (lgbm+knn) | **0.5517** | — |

**Key insight**: ChemBERTa-MLM has the worst individual RAE but the lowest residual correlation
with LGBM (ρ=0.728). It encodes different structural features — the meta-learner can down-weight
its noisy signal while still extracting its unique complementary patterns.

**Strategy**: 5-column OOF stack → RidgeCV meta-learner → nested scaffold CV RAE estimate.
Final blend with Chemprop (nb08, 0.5736) via inverse-RAE weights.

**Runtime**: ~15 min (all base OOF arrays are precomputed — just meta-learning).

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV, ElasticNetCV

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

plt.rcParams.update({"figure.dpi": 120})
print("Setup complete.")

In [ ]:
# ── 2. Load all OOF arrays and test predictions ────────────────────────────────
train = load_train()
te    = load_test()
y_tr  = train['pec50'].values

MODEL_NAMES = ['lgbm_base', 'lgbm_aug', 'knn', 'chemberta_mlm', 'chemberta_mtr']

oof_files = [
    DATA_PROCESSED / 'oof_lgbm_base.npy',
    DATA_PROCESSED / 'oof_lgbm_aug.npy',
    DATA_PROCESSED / 'oof_knn.npy',
    DATA_PROCESSED / 'oof_chemberta.npy',
    DATA_PROCESSED / 'oof_chemberta_mtr.npy',
]
te_files = [
    DATA_PROCESSED / 'te_lgbm_base.npy',
    DATA_PROCESSED / 'te_lgbm_aug.npy',
    DATA_PROCESSED / 'te_knn.npy',
    DATA_PROCESSED / 'te_chemberta.npy',
    DATA_PROCESSED / 'te_chemberta_mtr.npy',
]

oof_stack = np.column_stack([np.load(f) for f in oof_files])  # (4139, 5)
te_stack  = np.column_stack([np.load(f) for f in te_files])   # (513, 5)

print(f"OOF stack: {oof_stack.shape}  |  Test stack: {te_stack.shape}")
print()
print("Individual OOF RAEs:")
for name, col in zip(MODEL_NAMES, oof_stack.T):
    print(f"  {name:20s}: {rae_fn(y_tr, col):.4f}")

In [ ]:
# ── 3. Pairwise diversity analysis ────────────────────────────────────────────
resids = oof_stack - y_tr.reshape(-1, 1)

print("Pairwise Spearman correlation of OOF residuals:")
print(f"  {'':20s}", '  '.join(f'{n[:8]:>8}' for n in MODEL_NAMES))
for i, ni in enumerate(MODEL_NAMES):
    row = []
    for j, nj in enumerate(MODEL_NAMES):
        r = spearmanr(resids[:, i], resids[:, j]).statistic
        row.append(f'{r:8.3f}')
    print(f"  {ni:20s}", '  '.join(row))
print()
print("Lower = more diverse = more complementary signal for the ensemble.")

In [ ]:
# ── 4. Train RidgeCV meta-learner + nested scaffold CV ────────────────────────
scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=5, seed=42)
alphas    = np.logspace(-3, 4, 150)

# In-sample meta (for weight inspection)
meta_ridge = RidgeCV(alphas=alphas, cv=5)
meta_ridge.fit(oof_stack, y_tr)
print(f"Ridge alpha: {meta_ridge.alpha_:.4f}")
print(f"Ridge weights: { {n: round(float(w),3) for n, w in zip(MODEL_NAMES, meta_ridge.coef_)} }")
print(f"In-sample RAE:  {rae_fn(y_tr, meta_ridge.predict(oof_stack)):.4f}")
print()

# Nested scaffold CV — honest out-of-fold estimate
oof_nested = np.full(len(y_tr), np.nan)
for tr_idx, va_idx in splits:
    m = RidgeCV(alphas=alphas, cv=3)
    m.fit(oof_stack[tr_idx], y_tr[tr_idx])
    oof_nested[va_idx] = m.predict(oof_stack[va_idx])

nested_rae = rae_fn(y_tr, oof_nested)
print(f"Nested scaffold CV RAE: {nested_rae:.4f}")
print()
print("== Leaderboard ==")
print(f"  LGBM_aug (nb07):         0.5582")
print(f"  Meta-11 (LGBM+kNN):      0.5517")
print(f"  Grand ensemble (this):   {nested_rae:.4f}")

In [ ]:
# ── 5. Also try ElasticNet (L1+L2) — may zero out weak models ────────────────
oof_nested_en = np.full(len(y_tr), np.nan)
for tr_idx, va_idx in splits:
    m = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 1.0],
                     alphas=np.logspace(-4, 2, 50), cv=3,
                     max_iter=10000)
    m.fit(oof_stack[tr_idx], y_tr[tr_idx])
    oof_nested_en[va_idx] = m.predict(oof_stack[va_idx])

en_rae = rae_fn(y_tr, oof_nested_en)
# In-sample ElasticNet for weight inspection
meta_en = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 1.0],
                       alphas=np.logspace(-4, 2, 50), cv=5, max_iter=10000)
meta_en.fit(oof_stack, y_tr)
print(f"ElasticNet alpha: {meta_en.alpha_:.5f}  l1_ratio: {meta_en.l1_ratio_:.2f}")
print(f"ElasticNet weights: { {n: round(float(w),3) for n, w in zip(MODEL_NAMES, meta_en.coef_)} }")
print(f"ElasticNet nested CV RAE: {en_rae:.4f}")

# Use the better meta-learner
best_oof   = oof_nested    if nested_rae <= en_rae else oof_nested_en
best_meta  = meta_ridge    if nested_rae <= en_rae else meta_en
best_rae   = min(nested_rae, en_rae)
best_label = 'Ridge' if nested_rae <= en_rae else 'ElasticNet'
print(f"\nBest meta-learner: {best_label}  (nested CV RAE {best_rae:.4f})")

In [ ]:
# ── 6. Apply meta-learner to test set ─────────────────────────────────────────
grand_te = best_meta.predict(te_stack)
grand_te = np.clip(grand_te, y_tr.min() - 0.5, y_tr.max() + 0.5)

# Blend with Chemprop (nb08) via inverse-RAE weights
for fname in ['10_expanded_multitask.csv', '08_chemprop_cv_blend.csv']:
    cp_path = SUBMISSIONS / fname
    if cp_path.exists():
        cp_preds = pd.read_csv(cp_path).set_index('Molecule Name').loc[te['name'].values, 'pEC50'].values
        cp_src   = fname
        break

chemprop_rae = 0.5736
w_cp   = (1/chemprop_rae) / (1/chemprop_rae + 1/best_rae)
final_preds = w_cp * cp_preds + (1 - w_cp) * grand_te
final_preds = np.clip(final_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)

print(f"Chemprop source:   {cp_src}  (RAE {chemprop_rae:.4f})")
print(f"Grand ensemble RAE: {best_rae:.4f}  (meta: {best_label})")
print(f"Chemprop weight: {w_cp:.3f}  |  Grand ensemble weight: {1-w_cp:.3f}")
print(f"Final preds: {final_preds.min():.2f} – {final_preds.max():.2f}  (median {np.median(final_preds):.3f})")

In [ ]:
# ── 7. Diagnostic plot ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# OOF scatter
ax = axes[0]
ax.scatter(y_tr, best_oof, alpha=0.15, s=8, c='steelblue')
lo, hi = y_tr.min() - 0.2, y_tr.max() + 0.2
ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, alpha=0.7)
ax.set(xlabel='True pEC50', ylabel='Grand ensemble OOF',
       title=f'Grand ensemble OOF  (RAE={best_rae:.4f})')

# Model weights bar chart
ax = axes[1]
coef = best_meta.coef_
colors = ['steelblue' if c >= 0 else 'tomato' for c in coef]
ax.bar(range(len(MODEL_NAMES)), coef, color=colors, edgecolor='k', lw=0.5)
ax.set_xticks(range(len(MODEL_NAMES)))
ax.set_xticklabels([n.replace('_', '\n') for n in MODEL_NAMES], fontsize=8)
ax.set(ylabel='Meta-learner coefficient', title=f'{best_label} weights')
ax.axhline(0, color='k', lw=0.8)

# Prediction distribution
axes[2].hist(final_preds, bins=30, color='forestgreen', edgecolor='k', lw=0.4)
axes[2].set(xlabel='pEC50', title=f'Grand ensemble + Chemprop blend')

plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'figures' / '15_grand_ensemble.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 8. Save submission ────────────────────────────────────────────────────────
sub = pd.DataFrame({'Molecule Name': te['name'].values,
                    'SMILES':        te['smiles'].values,
                    'pEC50':         final_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '15_grand_ensemble.csv'
sub.to_csv(out, index=False)

np.save(DATA_PROCESSED / 'oof_grand15.npy',   best_oof)
np.save(DATA_PROCESSED / 'te_grand15.npy',    grand_te)

print(f"Saved: {out}")
print(f"Grand ensemble ({best_label}) nested CV RAE: {best_rae:.4f}")
print(f"Blend: {1-w_cp:.3f} × grand + {w_cp:.3f} × Chemprop")
print()
print("== Full leaderboard ==")
from pxr.eval import rae as rae_fn
lb = [
    ('Meta-11 (RidgeCV on LGBM+kNN)',   0.5517),
    ('Grand-15 (this notebook)',         best_rae),
    ('LGBM_aug (nb07)',                  0.5582),
    ('LGBM_base (nb11)',                 0.5600),
    ('ChemBERTa-MTR (nb14)',             0.5993),
    ('Chemprop 2-task (nb08)',           0.5736),
    ('ChemBERTa-MLM (nb13)',             0.6782),
    ('kNN k=20 (nb11 OOF)',              0.7341),
]
for name, r in sorted(lb, key=lambda x: x[1]):
    print(f"  {r:.4f}  {name}")
print(sub['pEC50'].describe().round(3))